## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## Load Dataset

In [2]:
df = pd.read_csv("Customer_Churn_Retention.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Basic Data Understanding

In [3]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Data Cleaning

#### Clean TotalCharges

In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


#### Add Calculated Columns

In [6]:
df["Churn_flag"] = df["Churn"].map({"Yes":1, "No":0})

df["Tenure_Group"] = pd.cut(
    df["tenure"],
    bins = [-1, 12, 24, 48, 100],
    labels = ["0-12 Months", "12-24 Months", "24-48 Months", "49+ Months"]
)

df["Monthly_Charge Band"] = pd.cut(
    df["MonthlyCharges"],
    bins = [0, 35, 70, 200],
    labels = ["Low Charges", "Medium Charges", "High Charges"]
)

df["Estimated_CLV"] = df["MonthlyCharges"] * df["tenure"]

## Exploratory Data Analysis

##### Churn Percentage

In [7]:
df["Churn"].value_counts(normalize = True) * 100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

##### Churn by Contract

In [8]:
df.groupby("Contract")["Churn_flag"].mean() * 100

Contract
Month-to-month    42.709677
One year          11.269518
Two year           2.831858
Name: Churn_flag, dtype: float64

##### Churn by PaymentMethod

In [9]:
df.groupby("PaymentMethod")["Churn_flag"].mean().sort_values(ascending= False) * 100

PaymentMethod
Electronic check             45.285412
Mailed check                 19.106700
Bank transfer (automatic)    16.709845
Credit card (automatic)      15.243101
Name: Churn_flag, dtype: float64

##### Churn by InternetService

In [10]:
df.groupby("InternetService")["Churn_flag"].mean().sort_values(ascending= False) * 100

InternetService
Fiber optic    41.892765
DSL            18.959108
No              7.404980
Name: Churn_flag, dtype: float64

##### Churn by TechSupport

In [11]:
df.groupby("TechSupport")["Churn_flag"].mean().sort_values(ascending= False) * 100

TechSupport
No                     41.635474
Yes                    15.166341
No internet service     7.404980
Name: Churn_flag, dtype: float64

##### Churn by Tenure_Group

In [12]:
df.groupby("Tenure_Group", observed = True)["Churn_flag"].mean() * 100

Tenure_Group
0-12 Months     47.438243
12-24 Months    28.710938
24-48 Months    20.388959
49+ Months       9.513176
Name: Churn_flag, dtype: float64

##### Churn by Monthly_Charge_Band

In [13]:
df.groupby("Monthly_Charge Band", observed = True)["Churn_flag"].mean() * 100

Monthly_Charge Band
Low Charges       10.893372
Medium Charges    23.942029
High Charges      35.361429
Name: Churn_flag, dtype: float64

## Prepare Model Data

In [14]:
model_df = df.drop(columns=["customerID"])

for col in model_df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])

model_df["Tenure_Group"] = model_df["Tenure_Group"].cat.codes
model_df["Monthly_Charge Band"] = model_df["Monthly_Charge Band"].cat.codes

## Train_test Split

In [15]:
X = model_df.drop(columns=["Churn","Churn_flag"])
y = model_df["Churn_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
) 

## Logistic Regression Model

In [16]:
log_model = LogisticRegression(max_iter = 10000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

## Random Forest Model

In [17]:
rf_model = RandomForestClassifier(
    n_estimators = 200,
    random_state = 42,
    class_weight = "balanced"
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

## Model Evaluation

In [18]:
print("Accuracy:",accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))

print(classification_report(y_test, y_pred_rf))

Accuracy: 0.794889992902768
Precision: 0.6534296028880866
Recall: 0.4839572192513369
F1 Score: 0.5560675883256528
ROC AUC: 0.8247797669792555
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1035
           1       0.65      0.48      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



## Churn Probability

In [19]:
df["Churn_Probability"] = rf_model.predict_proba(X)[:, 1]

## Risk Segment

In [20]:
df["Risk_Segment"] = pd.cut(
    df["Churn_Probability"],
    bins = [0, 0.30, 0.60, 1],
    labels = ["Low Risk", "Medium Risk", "High Risk"]
)

## Recommended Action

In [21]:
def Recommend_Action(row):
    if row["Risk_Segment"] == "High Risk" and row["Contract"] == "Month-to-month":
        return "Offer annual contract discount"
    elif row["TechSupport"] == "No" and row["InternetService"] == "No":
        return "Offer free tech support trial"
    elif row["PaymentMethod"] == "Electonic check":
        return "Promote auto-payment method"
    elif row["MonthlyCharges"] > 70:
        return "Offer loyalty discount"
    else:
        return "Monitor customer"

df["Recommended_Action"] = df.apply(Recommend_Action, axis = 1)

## Export Output

In [22]:
df.to_csv("Customer_Churn_Risk_output.csv", index = False)